<a href="https://colab.research.google.com/github/daria-bazaliy/Kaggle-competitions/blob/main/safe_driver_risk_prediction_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

baseline

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

# 1. Загрузка
train = pd.read_csv('train.csv')
X = train.drop(['id', 'target'], axis=1)
y = train['target']

# 2. Разделение
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ВАРИАНТ 1: Баланс между простотой и эффективностью
model_simple = lgb.LGBMClassifier(
    n_estimators=1000,  # БОЛЬШЕ деревьев, но с early stopping
    scale_pos_weight=25,  # примерный дисбаланс (1/0.0364 ≈ 27.5)
    learning_rate=0.05,   # умеренная скорость
    num_leaves=31,       # стандартно
    min_child_samples=20, # стандартно
    reg_alpha=0.1,       # небольшая регуляризация
    reg_lambda=0.1,      # небольшая регуляризация
    subsample=0.8,       # защита от переобучения
    colsample_bytree=0.8, # защита от переобучения
    random_state=42,
    verbose=-1
)

# Обучаем с ранней остановкой
model_simple.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='binary_logloss',
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(0)
    ]
)

# Получаем предсказания
y_pred_proba_simple = model_simple.predict_proba(X_val)[:, 1]

# Ищем лучший порог для F1
best_f1_simple = 0
best_threshold_simple = 0.5
thresholds = np.linspace(0.001, 0.5, 200)  # больше точек для поиска

for thresh in thresholds:
    y_pred = (y_pred_proba_simple >= thresh).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1_simple:
        best_f1_simple = f1
        best_threshold_simple = thresh

print(f"Простая модель:")
print(f"Лучший порог: {best_threshold_simple:.4f}")
print(f"Лучший F1: {best_f1_simple:.4f}")

test = pd.read_csv("test.csv")

test_ids = test["id"]

X_test = train.drop(['id', 'target'], axis=1).copy()

test_pred_proba = model_simple.predict_proba(X_test)[:, 1]

test_pred = (test_pred_proba >= best_threshold_simple).astype(int)

submission = pd.DataFrame({
    "id": test_ids,
    "target": test_pred
})

submission.to_csv("sub_baseline.csv", index=False)

print("submission.csv сохранён")
print(submission.head())


основная модель

In [ ]:
from sklearn.feature_selection import mutual_info_classif
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

# Загрузка данных
train = pd.read_csv('train.csv')

def categorize_features_simple(df, target_col='target', max_categories=20):
    """
    Простая и понятная категоризация:
    - Бинарные: только {0, 1} или {0, 1, -1}
    - Категориальные: целые числа, мало уникальных (≤ max_categories)
    - Числовые: всё остальное (кроме id, missing, target)
    """
    X = df.drop(columns=[target_col], errors='ignore')

    # 1. Сначала исключим служебные колонки
    exclude_cols = []
    if 'id' in X.columns:
        exclude_cols.append('id')
    if 'missing' in X.columns:
        exclude_cols.append('missing')

    # 2. Бинарные колонки
    binary_cols = []

    for col in X.columns:
        if col in exclude_cols:
            continue

        unique_vals = df[col].dropna().unique()

        # Проверяем, что значения только из {-1, 0, 1}
        if set(unique_vals).issubset({-1, 0, 1}):
            # И что это не все три значения (только 0/1 или с -1)
            if len(unique_vals) in [2, 3]:
                binary_cols.append(col)

    print(f"1. Найдено бинарных колонок: {len(binary_cols)}")

    # 3. Категориальные колонки
    categorical_cols = []

    for col in X.columns:
        if col in exclude_cols or col in binary_cols:
            continue

        # Проверяем, что тип целочисленный
        if df[col].dtype in ['int64', 'int32', 'int16', 'int8', 'uint8', 'category']:
            n_unique = df[col].nunique()

            # Проверяем, что уникальных значений немного
            if 2 < n_unique <= max_categories:  # >2 чтобы не пересекаться с бинарными
                categorical_cols.append(col)

    print(f"2. Найдено категориальных колонок (≤{max_categories} уникальных): {len(categorical_cols)}")

    # 4. Числовые колонки
    numeric_cols = []

    for col in X.columns:
        if (col not in exclude_cols and
            col not in binary_cols and
            col not in categorical_cols and
            col != target_col):
            numeric_cols.append(col)

    print(f"3. Найдено числовых колонок: {len(numeric_cols)}")

    # 5. Проверка
    total_cols = len(binary_cols) + len(categorical_cols) + len(numeric_cols) + len(exclude_cols)
    print(f"\nВсего колонок: {total_cols} (исходно: {len(X.columns)})")

    if total_cols != len(X.columns):
        print("⚠️ Внимание: некоторые колонки не попали ни в одну категорию!")
        missing = set(X.columns) - set(binary_cols + categorical_cols + numeric_cols + exclude_cols)
        print(f"Пропущенные колонки: {list(missing)}")

    return binary_cols, categorical_cols, numeric_cols, exclude_cols

# Тестируем функцию
print("="*50)
print("КАТЕГОРИЗАЦИЯ ПРИЗНАКОВ")
print("="*50)

binary_cols, categorical_cols, numeric_cols, exclude_cols = categorize_features_simple(train, max_categories=20)

# Покажем примеры
print("\nПримеры колонок каждого типа:")
print(f"Бинарные (первые 5): {binary_cols[:5]}")
print(f"Категориальные (первые 5): {categorical_cols[:5]}")
print(f"Числовые (первые 5): {numeric_cols[:5]}")
print(f"Служебные: {exclude_cols}")


def select_top_binary_features(
    X, y, binary_cols, top_k=40, random_state=42
):
    """
    Жёстко отбирает top_k бинарных фичей по MI с таргетом
    """
    if len(binary_cols) <= top_k:
        print(f"Бинарных фичей мало ({len(binary_cols)}), ничего не режем")
        return binary_cols

    X_bin = X[binary_cols].copy()

    # заменяем -1, чтобы MI считался корректно
    X_bin = X_bin.replace(-1, 0)

    mi = mutual_info_classif(
        X_bin,
        y,
        discrete_features=True,
        random_state=random_state
    )

    mi_scores = pd.Series(mi, index=binary_cols)\
                  .sort_values(ascending=False)

    selected = mi_scores.head(top_k).index.tolist()

    print(f"Бинарные фичи: {len(binary_cols)} → {len(selected)}")
    return selected


binary_cols_sel = select_top_binary_features(
    train.drop(columns=["target"]),
    train["target"],
    binary_cols,
    top_k=40
)

final_features = (
    binary_cols_sel +
    categorical_cols +
    numeric_cols
)

X_final = train[final_features]
y = train["target"]

# 2. Разделение
X_train, X_val, y_train, y_val = train_test_split(
    X_final, y, test_size=0.2, random_state=42, stratify=y
)

# ВАРИАНТ 1: Баланс между простотой и эффективностью
model_simple = lgb.LGBMClassifier(
    n_estimators=1000,  # БОЛЬШЕ деревьев, но с early stopping
    scale_pos_weight=25,  # примерный дисбаланс (1/0.0364 ≈ 27.5)
    learning_rate=0.05,   # умеренная скорость
    num_leaves=31,       # стандартно
    min_child_samples=20, # стандартно
    reg_alpha=0.1,       # небольшая регуляризация
    reg_lambda=0.1,      # небольшая регуляризация
    subsample=0.8,       # защита от переобучения
    colsample_bytree=0.8, # защита от переобучения
    random_state=42,
    verbose=-1
)

# Обучаем с ранней остановкой
model_simple.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='binary_logloss',
    callbacks=[
        lgb.early_stopping(50, verbose=False),
        lgb.log_evaluation(0)
    ]
)

# Получаем предсказания
y_pred_proba_simple = model_simple.predict_proba(X_val)[:, 1]

# Ищем лучший порог для F1
best_f1_simple = 0
best_threshold_simple = 0.5
thresholds = np.linspace(0.001, 0.5, 200)  # больше точек для поиска

for thresh in thresholds:
    y_pred = (y_pred_proba_simple >= thresh).astype(int)
    f1 = f1_score(y_val, y_pred)
    if f1 > best_f1_simple:
        best_f1_simple = f1
        best_threshold_simple = thresh

print(f"Простая модель:")
print(f"Лучший порог: {best_threshold_simple:.4f}")
print(f"Лучший F1: {best_f1_simple:.4f}")


test = pd.read_csv("test.csv")

test_ids = test["id"]

X_test = test[final_features].copy()

test_pred_proba = model_simple.predict_proba(X_test)[:, 1]

test_pred = (test_pred_proba >= best_threshold_simple).astype(int)

submission = pd.DataFrame({
    "id": test_ids,
    "target": test_pred
})

submission.to_csv("subf1.csv", index=False)

print("submission.csv сохранён")
print(submission.head())